# MolmoAct2 LoRA fine-tuning workshop (AMD, ROCm)

This notebook takes a **pretrained MolmoAct2** vision-language-action model and **LoRA fine-tunes** it, then evaluates the fine-tuned policy in the **headless LIBERO simulator** - all on AMD hardware (a single Strix Halo iGPU, or a multi-GPU AMD Instinct node).

**Real -> sim story.** We start from `allenai/MolmoAct2-DROID` (trained on the *real* Franka Panda arm, delta end-effector control) and LoRA-adapt it onto the `allenai/MolmoAct2-LIBERO-Dataset` (the LIBERO *simulator*). Same Panda embodiment and action space, so the base is well-initialized; the LoRA adapter learns the sim.

**How LoRA works here.** The heavy VLM backbone (SigLIP2 vision tower + Qwen3-class LM) is *frozen*; small low-rank adapter matrices are trained on top. The flow-matching action expert stays fully trainable (upstream found this crucial). This keeps memory low enough to fine-tune on a single Strix Halo.

Steps:
1. **Model import smoke-test** - ROCm/GPU sanity + import the MolmoAct2 stacks
2. **Download + load** the base checkpoint (one real forward proves the load)
3. **Open-loop rollout** on a few real DROID episodes (GT-vs-pred overlay)
4. **LoRA fine-tune** setup + a few steps (auto-scales single/multi-GPU)
5. **Load the LoRA checkpoint on the DROID base -> LIBERO policy** + closed-loop eval

The **teaching code stays inline** - steps 1-3 (probe, load, open-loop) run directly *in the kernel*, and the fine-tune / eval cells build their exact command inline so you can read every flag. The bulky, non-teaching **plumbing** (subprocess streaming, Hugging Face download management, and DROID video decoding) lives in `scripts/ft_helpers.py`, imported at the top, so the cells stay focused on the concepts. ROCm's noisy `HIPBLAS`/experimental-attention warnings are suppressed so the output stays readable.

This runs on a single Strix Halo iGPU; the sibling `finetune_molmoact2_cluster.ipynb` retargets step 4 to a multi-GPU AMD Instinct node.

## Goals

* Understand **LoRA fine-tuning** of a vision-language-action model: what stays *frozen* (the SigLIP2 vision tower + Qwen3-class language model), what *trains* (small low-rank adapters plus the flow-matching action expert), and why that keeps memory low enough to run on a single Strix Halo iGPU
* Follow the **real-to-sim** story: start from a policy trained on a *real* Franka Panda arm and adapt it to the LIBERO *simulator*, which shares the same embodiment and action space so the base is already well-initialized
* Run the full pipeline end to end - import check, load the base, open-loop sanity check, fine-tune, closed-loop eval - entirely on AMD ROCm hardware
* Learn which knobs (`STEPS`, `BATCH_SIZE`, `FT_MODE`) take this from a workshop smoke-test to a real training run

## How this fits together

| Piece | Role |
| --- | --- |
| **MolmoAct2-DROID** | The base policy, trained on the *real* Franka arm - the starting point we adapt |
| **MolmoAct2-LIBERO dataset** | The simulator demonstrations the LoRA adapter learns from |
| **LoRA fine-tune** | Trains small low-rank adapter matrices on the frozen backbone, plus the flow-matching action expert |
| **LIBERO simulator** | Headless MuJoCo (EGL on the AMD GPU) where the fine-tuned policy is scored |

The numbered steps below walk this path once: a fast import/environment check, load the base and prove one forward pass, sanity-check it open-loop on real data, run the LoRA fine-tune, then evaluate the result closed-loop in the simulator.

In [ ]:
import os

# --- Quiet ROCm/torch on this iGPU *before* importing torch --------------------------------
# On gfx1151 every matmul otherwise logs "HIPBLAS_STATUS_NOT_SUPPORTED ... will attempt to
# recover by calling cublas" and "experimental flash/mem-efficient attention" UserWarnings,
# which flood the notebook. Preferring plain hipBLAS over hipBLASLt stops the fallback spam
# at the source; PYTHONWARNINGS + filterwarnings hide the rest (in-kernel and in children).
os.environ.setdefault("TORCH_BLAS_PREFER_HIPBLASLT", "0")
os.environ.setdefault("PYTHONWARNINGS", "ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
# Silence the per-file download bars (file-003.parquet 100% ...) and the tqdm train/eval/load
# loops that otherwise redraw every fraction of a second and swamp the cell output. Progress is
# instead reported by the loaders' own periodic log lines (and the loss plot in Step 4).
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TQDM_DISABLE", "1")
# Show a plain GB-progress heartbeat for the big HF pulls (env-independent, no tqdm).
os.environ.setdefault("VERBOSE_DOWNLOAD", "1")

import glob
import json
import logging
import sys
import time
import warnings

warnings.filterwarnings("ignore")
# Silence per-request HTTP logs and the base loader's benign bf16-patch "needle not found"
# notices so the only thing printed is the workshop's own progress.
for _n in (
    "transformers",
    "lerobot",
    "accelerate",
    "datasets",
    "huggingface_hub",
    "httpx",
    "httpcore",
    "urllib3",
    "host_server_droid",
):
    logging.getLogger(_n).setLevel(logging.ERROR)

import numpy as np
import torch

# Paths inside the workshop image. The teaching code runs in THIS kernel's trainable venv; the
# bulky, non-teaching plumbing (subprocess streaming, HF download management, DROID video decode)
# lives in scripts/ft_helpers.py so the cells stay readable. The genuine subprocesses left
# (fine-tune / eval) still build their exact command inline in the cell below.
RYZERS = "/ryzers"
DROID_SRV = os.environ.get("DROID_SERVER_DIR", "/repos/molmoact2/examples/droid")
TRAIN_PY = "/opt/train-venv/bin/python"
OUT_DIR = os.environ.get("OUT_DIR", "/outputs")
os.makedirs(OUT_DIR, exist_ok=True)
if DROID_SRV not in sys.path:
    sys.path.insert(0, DROID_SRV)

# Make the workshop helper module importable, then pull in the plumbing used below.
_SCRIPTS = os.path.join(RYZERS, "notebooks", "scripts")
if _SCRIPTS not in sys.path:
    sys.path.insert(0, _SCRIPTS)
from ft_helpers import (
    fetch_full_libero,
    preflight_inputs,
    prefetch,
    run_openloop_episodes,
    run_train,
    stage_assets,
    stream_cmd,
)


# Env for child processes: keep the warning-suppression flags and DROP the inherited
# matplotlib-inline backend (MPLBACKEND=module://matplotlib_inline...), which crashes
# headless children (the open-loop / eval crash we hit before).
def child_env(**extra):
    e = dict(os.environ)
    e.pop("MPLBACKEND", None)
    # Let the training/eval CHILDREN render their own tqdm progress bars (the kernel keeps tqdm
    # disabled so in-notebook code stays quiet). stream_cmd/run_train preserve carriage returns, so
    # these bars redraw on ONE line instead of spamming a new line per tick.
    e.pop("TQDM_DISABLE", None)
    e.update({"TORCH_BLAS_PREFER_HIPBLASLT": "0", "PYTHONWARNINGS": "ignore", "PYTHONUNBUFFERED": "1"})
    e["PYTHONPATH"] = _SCRIPTS + (":" + e["PYTHONPATH"] if e.get("PYTHONPATH") else "")
    e.update({k: str(v) for k, v in extra.items()})
    return e


# Flow-matching action-head call, tolerant of the inference_action_mode (new) vs action_mode
# (old) kwarg rename across MolmoAct2 checkpoints. Kept inline so the model-call semantics stay
# visible next to the Step 2-3 smoke-test and open-loop rollout that use it.
def predict_chunk(policy, norm_tag, images, task, state, num_steps):
    kw = {
        "processor": policy.processor,
        "images": images,
        "task": task,
        "state": state,
        "norm_tag": norm_tag,
        "enable_depth_reasoning": False,
        "num_steps": num_steps,
        "normalize_language": True,
        "enable_cuda_graph": False,
    }
    try:
        out = policy.model.predict_action(inference_action_mode="continuous", **kw)
    except TypeError as e:
        if "action_mode" not in str(e):
            raise
        out = policy.model.predict_action(action_mode="continuous", **kw)
    raw = out.actions if hasattr(out, "actions") else out
    if torch.is_tensor(raw):
        raw = raw.detach().to(dtype=torch.float32, device="cpu").numpy()
    a = np.asarray(raw, dtype=np.float32)
    if a.ndim == 3 and a.shape[0] == 1:
        a = a[0]
    return a


## 1. Model import smoke-test

Every step that follows needs a working ROCm GPU and the training stack importable in this kernel, so we check that first - before any slow downloads or model loads - so a broken environment fails here in seconds instead of ten minutes later.

Confirms the trainable venv sees a ROCm device and imports the LeRobot MolmoAct2 training stack (torch+HIP, transformers, lerobot, robosuite/mujoco). The detected **GPU count** drives sensible defaults below (small batch/steps on a single Strix Halo, larger on a multi-GPU Instinct node). No weights are loaded here - this is a fast import/environment check.

In [ ]:
# In-kernel probe (no subprocess): confirm a ROCm device and that the trainable MolmoAct2
# + sim stack imports. The detected GPU count drives the batch/step defaults below.
assert torch.version.hip, "torch is not a ROCm build"
assert torch.cuda.is_available(), "no ROCm device visible (check /dev/kfd, /dev/dri)"
print(f"torch            : {torch.__version__}  hip={torch.version.hip}")

N_GPUS, gpu_names, vram_gib = max(1, torch.cuda.device_count()), [], []
for i in range(torch.cuda.device_count()):
    pr = torch.cuda.get_device_properties(i)
    gpu_names.append(pr.name)
    vram_gib.append(round(pr.total_memory / 1024**3, 1))
    print(f"device[{i}]        : {pr.name}  ({vram_gib[-1]} GiB)")

for mod in ("lerobot", "transformers", "robosuite", "mujoco", "accelerate"):
    m = __import__(mod)
    print(f"{mod:16s} : {getattr(m, '__version__', '?')}")

print(f"\n==> detected {N_GPUS} GPU(s): {gpu_names}  VRAM(GiB)={vram_gib}")

## 2. Download + load the base checkpoint

Downloads the base checkpoint to adapt (`allenai/MolmoAct2-DROID`, real Franka) and the fine-tuning dataset (`allenai/MolmoAct2-LIBERO-Dataset`) into the persistent HF cache, then **loads the checkpoint and runs one real forward** to prove the whole flow-matching action path executes on ROCm (bf16). Nothing is re-hosted by us; everything pulls from Hugging Face at run time. Set `HF_TOKEN` for faster/gated downloads. First run pulls tens of GB - it is cached and reused afterwards.

**One base, used twice (and how the assets are packaged).** This exact same `allenai/MolmoAct2-DROID` model is reused for *both* the open-loop check (Step 3, loaded here) *and* as the starting point of the LoRA fine-tune (Step 4 passes `--policy.checkpoint_path=allenai/MolmoAct2-DROID`) - it is loaded once and shared from the HF cache. We load it and train in **bf16**. Downloaded fresh from the Hub it arrives in **fp32** (~22 GB across 5 shards ≈ 4 bytes × 5.4B params); the pre-staged workshop assets instead ship it already in **bf16** (~11 GB), the precision we actually run in. LeRobot saves a fine-tuned policy as a single bf16 `model.safetensors` (~11.5 GB) plus tiny normalizer stats - the frozen base with the trained update on top. To avoid shipping the base twice, our ready-made reference checkpoint is packaged **split**: the bf16 base plus a small trained *delta* (the LoRA adapter + trained action-expert, ~2.4 GB), rebuilt into that same full checkpoint automatically when the assets are staged (`fetch_assets.sh`).

In [ ]:
BASE_CKPT = os.environ.get("BASE_CKPT", "allenai/MolmoAct2-DROID")
DATASET_REPO = os.environ.get("DATASET_REPO", "allenai/MolmoAct2-LIBERO-Dataset")

# Optional: if the workshop hosts the base checkpoint + dataset + fine-tuned checkpoint on local
# storage, point ASSETS_DIR at that folder and they are copied into the HF cache (no Hub
# re-download). Otherwise the image-baked cache is used as-is. Idempotent.
stage_assets()

# Confirm every input is present BEFORE the long model load / training, so a missing asset fails
# fast with a clear message instead of deep inside the loader.
preflight_inputs(BASE_CKPT, DATASET_REPO)

# Fetch the base checkpoint into the HF cache (skips instantly if already cached).
prefetch(BASE_CKPT, "model")

# LIBERO dataset routing. DEFAULT: the small pre-staged SUBSET (offline, ~1 GB) so the workshop
# payload stays tiny and the short Step-4 fine-tune runs on it. The subset is a self-consistent
# LeRobot dataset (metadata rewritten to only the bundled episodes), so training "just works".
# EXTENDED ROUTE: set USE_FULL_LIBERO=1 to pull the COMPLETE dataset from the Hub (needs internet;
# ~33 GB). Because the staged subset reuses each file\'s real content hash, only missing files
# download.
if os.environ.get("USE_FULL_LIBERO", "0") == "1":
    fetch_full_libero(DATASET_REPO)
else:
    prefetch(DATASET_REPO, "dataset")
print("PASS: base checkpoint + dataset cached")


In [ ]:
# Load the base checkpoint in THIS kernel via the upstream DROID Policy loader (bf16 patches
# baked at /repos/molmoact2/examples/droid) and run ONE real forward. Proves the full 6B
# flow-matching action path executes on ROCm and that the just-downloaded weights load. First
# call may JIT-compile kernels (slow once). We keep `base_policy` around to reuse in Step 3.
sys.path.insert(0, "scripts")
from fast_to_device import install
install()
from host_server_droid import NORM_TAG, Policy
from PIL import Image

DTYPE = {"bfloat16": torch.bfloat16, "float16": torch.float16, "float32": torch.float32}[
    os.environ.get("DTYPE", "bfloat16")
]
NUM_STEPS = int(os.environ.get("NUM_STEPS", "10"))

if globals().get("base_policy") is None:
    t0 = time.time()
    base_policy = Policy(repo_id=BASE_CKPT, device="cuda:0", dtype=DTYPE)
    n_params = sum(p.numel() for p in base_policy.model.parameters())
    print(f"model loaded     : {time.time() - t0:.1f}s  params={n_params / 1e9:.2f}B  norm_tag={NORM_TAG}")
else:
    print("base_policy already loaded in this kernel; reusing it (restart the kernel to force a reload)")

# One dummy DROID observation: 3 cams (ext1, ext2, wrist) + 8-DoF state.
imgs = [Image.fromarray(np.random.randint(0, 255, (180, 320, 3), dtype=np.uint8)) for _ in range(3)]
t1 = time.time()
acts = predict_chunk(base_policy, NORM_TAG, imgs, "pick up the object", np.zeros(8, np.float32), NUM_STEPS)
print(f"predict_action   : {acts.shape}  in {(time.time() - t1) * 1000:.0f} ms")
assert acts.ndim == 2 and acts.shape[-1] == 8 and np.isfinite(acts).all(), f"bad actions {acts.shape}"
print("PASS: MolmoAct2 full-model ROCm smoke OK")

## 3. Open-loop rollout on a few real DROID episodes

An **open-loop rollout** replays a recorded episode and, at each replan step, compares the actions the policy *predicts* against the actions the human teleoperator actually took - it never feeds the predictions back into the simulator, so errors don't compound. If the base can't reproduce the data it was trained on, nothing downstream will, which makes this the cheapest sanity check to run before we spend time fine-tuning.

Before fine-tuning, sanity-check the base policy on the **real** data it was trained on. This is a multi-episode port-fidelity check: we draw **`N_DROID_EPISODES` random episodes** (default `2`) from `allenai/MolmoAct2-DROID-Dataset` - each is a *different* real teleop episode with its own task string - and replay each one open-loop (receding-horizon: replan every `STRIDE` steps). For every episode we write, under `/outputs`:

- `droid_ep<E>.mp4` - the episode's exterior camera view (the scene the model predicts on),
- `droid_ep<E>_actions.png` - the 8-DoF **GT (teleop) vs predicted** action trajectory, overlaid per dim on the same axes (GT solid, pred dashed).

**About the downloads you see.** MolmoAct2 consumes **3 camera streams** (`exterior_1`, `exterior_2`, `wrist`), so each random episode pulls its own 3 camera `.mp4` clips (plus one small `.parquet` of states/actions) on demand from the Hub. That is why you see downloads continue per episode, and why - mid-run - the file count can be ahead of the finished-episode count (e.g. the 3rd clip of episode 2 is still streaming while only episode 1 has printed its metrics). It is **bounded**: total clips ≈ `3 × N_DROID_EPISODES`, and everything is cached for reruns. This is *not* the closed-loop task benchmark (that's Step 5); to make it lighter set `N_DROID_EPISODES=1`, or pin one episode with `EPISODE=<id>`.

The newest episode's plot + video are shown inline below.

In [ ]:
# Replay a few real DROID episodes open-loop: replan every STRIDE steps and overlay the
# ground-truth teleop actions (solid) against MolmoAct2\'s predictions (dashed), per dim, with
# the exterior-cam video - all rendered inline. `predict_chunk` (defined above) is passed in so
# the flow-matching action-head call stays visible; the video-decode machinery lives in the helper.
run_openloop_episodes(base_policy, predict_chunk, norm_tag=NORM_TAG, num_steps=NUM_STEPS, out_dir=OUT_DIR)

# Release the base DROID policy so the fine-tune below has the iGPU to itself.
import gc

del base_policy
gc.collect()
torch.cuda.empty_cache()


## 4. LoRA fine-tune (setup + a few steps)

This is the heart of the notebook: we adapt the base policy to the LIBERO simulator by training small **LoRA** adapter matrices on top of the frozen backbone (plus the action expert), rather than updating all ~5.4B weights. That is what makes fine-tuning a model this size feasible on a single iGPU. A short run here just proves the training loop works end to end - the loss curve below should already trend downward.

The same command runs single-GPU on Strix Halo and multi-GPU on an AMD Instinct node (`accelerate` uses `N_GPUS` processes automatically; the cluster notebook shows the multi-GPU path).

**Fine-tune modes** (`FT_MODE`, maps to `--policy.train_mode_vlm`):
- `lora_vlm` *(default)* - LoRA adapters on the VLM; action expert fully trainable. ~20 GiB at batch 8.
- `action_expert_only` - freeze the VLM, train only the flow-matching action expert. ~16 GiB at batch 8.
- `full` - full fine-tune of VLM + action expert (~48 GiB at batch 8; prefer a multi-GPU Instinct node).

For the workshop we run only a **few steps** to see the loop work end to end; a longer-trained *reference* checkpoint is provided for a compelling sim demo (see the last section).

In [ ]:
import shutil

FT_MODE = os.environ.get("FT_MODE", "lora_vlm")
# Workshop default: just 10 steps to prove the loop end to end. SAVE_FREQ defaults to STEPS,
# so a checkpoint reliably lands at the final step (this is what Step 5 loads). Raise STEPS
# (e.g. 10000) for real training.
STEPS = int(os.environ.get("STEPS", "10"))
SAVE_FREQ = int(os.environ.get("SAVE_FREQ", str(STEPS)))
# Log every step by default so the short workshop run yields a full loss curve (one point per
# step) to plot below; bump for long runs where per-step logging is too chatty.
LOG_FREQ = int(os.environ.get("LOG_FREQ", "1"))
# Per-GPU batch size: keep small on a single Strix Halo, larger with more GPUs.
BATCH_SIZE = int(os.environ.get("BATCH_SIZE", "2" if N_GPUS == 1 else "8"))
JOB_NAME = os.environ.get("JOB_NAME", f"mm2_{FT_MODE}_ws")
CKPT_DIR = os.environ.get("CHECKPOINTS_DIR", "/checkpoints")
OUTPUT_DIR = os.path.join(CKPT_DIR, JOB_NAME)
# The allenai LIBERO dataset ships without LeRobot codebase-version tags; pin a branch.
DATASET_REVISION = os.environ.get("DATASET_REVISION", "main")

# FT_MODE -> --policy.train_mode_vlm : lora (adapters on VLM + trainable action expert),
# freeze (VLM frozen, action expert only), fft (full fine-tune).
MODE_ARGS = {
    "lora_vlm": ["--policy.train_mode_vlm=lora", "--policy.action_mode=both"],
    "action_expert_only": ["--policy.train_mode_vlm=freeze", "--policy.action_mode=continuous"],
    "full": ["--policy.train_mode_vlm=fft", "--policy.action_mode=both"],
}[FT_MODE]

# LeRobot refuses to write into a non-empty output_dir; clean prior workshop reruns.
if os.path.isdir(OUTPUT_DIR) and os.environ.get("CLEAN_OUTPUT", "1") == "1":
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(CKPT_DIR, exist_ok=True)

print("=" * 63)
print(f" MolmoAct2 fine-tune | base={BASE_CKPT}  mode={FT_MODE}")
print(f"   GPUs(procs)={N_GPUS}  batch/GPU={BATCH_SIZE}  steps={STEPS}  save_freq={SAVE_FREQ}")
print(f"   output_dir={OUTPUT_DIR}")
print("=" * 63)

# The exact accelerate + LeRobot training command, shown inline (no hidden wrapper). The SAME
# command runs single-GPU on Strix Halo and multi-GPU on an Instinct node (num_processes=N_GPUS).
cmd = [
    TRAIN_PY,
    "-m",
    "accelerate.commands.launch",
    f"--num_processes={N_GPUS}",
    "--mixed_precision=bf16",
    "-m",
    "lerobot.scripts.lerobot_train",
    f"--dataset.repo_id={DATASET_REPO}",
    f"--dataset.revision={DATASET_REVISION}",
    "--dataset.video_backend=pyav",
    "--dataset.image_transforms.enable=true",
    "--policy.type=molmoact2",
    f"--policy.checkpoint_path={BASE_CKPT}",
    "--policy.device=cuda",
    *MODE_ARGS,
    "--policy.chunk_size=10",
    "--policy.n_action_steps=10",
    "--policy.setup_type=single franka robotic arm in libero",
    "--policy.control_mode=delta end-effector pose",
    '--policy.image_keys=["observation.images.image","observation.images.wrist_image"]',
    "--policy.model_dtype=bfloat16",
    "--policy.num_flow_timesteps=8",
    "--policy.gradient_checkpointing=true",
    "--policy.freeze_embedding=true",
    "--policy.normalize_gripper=false",
    "--policy.enable_knowledge_insulation=false",
    "--policy.push_to_hub=false",
    f"--wandb.enable={os.environ.get('WANDB_ENABLE', 'false')}",
    f"--job_name={JOB_NAME}",
    f"--output_dir={OUTPUT_DIR}",
    f"--steps={STEPS}",
    f"--batch_size={BATCH_SIZE}",
    f"--num_workers={os.environ.get('NUM_WORKERS', '4')}",
    f"--log_freq={LOG_FREQ}",
    f"--eval_freq={STEPS + 1}",
    "--save_checkpoint=true",
    f"--save_freq={SAVE_FREQ}",
]
loss_pts = run_train(cmd, env=child_env())
print(f"PASS: fine-tune finished; checkpoints under {OUTPUT_DIR}/checkpoints/")

# Loss curve for the run we just did (LeRobot logs one point per --log_freq step). Even a short
# workshop run should trend downward; raise STEPS for a smoother, more convincing curve.
import matplotlib.pyplot as plt  # inline backend -> renders in the notebook, no Agg/subprocess

if len(loss_pts) >= 2:
    steps, losses = zip(*loss_pts)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(steps, losses, marker="o", color="tab:red", lw=1.6)
    ax.set_xlabel("training step")
    ax.set_ylabel("loss")
    ax.set_title(f"LoRA fine-tune loss ({FT_MODE}, {STEPS} steps, batch {BATCH_SIZE}×{N_GPUS})")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "finetune_loss.png"), dpi=110)
    plt.show()
    print(f"loss: {losses[0]:.4f} -> {losses[-1]:.4f} over {len(losses)} logged step(s)")
else:
    print(f"(captured {len(loss_pts)} loss point(s) - need >=2 to plot; raise STEPS or lower LOG_FREQ)")

In [ ]:
# Which checkpoint do Step 5 (closed-loop eval) and Step 6 (interactive sim) load?
# DEFAULT: our fine-tuned LIBERO checkpoint, loaded DIRECTLY - so you get a strong policy
# without waiting on a long train. It lives at REFERENCE_POLICY (staged onto pod storage).
# Overrides: POLICY_PATH=/path/or/hub-id wins; PREFER_TRAINED=1 evaluates the checkpoint the
# short Step-4 run just produced instead.
REFERENCE = os.environ.get(
    "REFERENCE_POLICY", os.path.join(CKPT_DIR, "reference", "pretrained_model")
)
_explicit = os.environ.get("POLICY_PATH", "").strip()
_prefer_trained = os.environ.get("PREFER_TRAINED", "0") == "1"
_trained = sorted(glob.glob(os.path.join(OUTPUT_DIR, "checkpoints", "*", "pretrained_model")))
_trained = [c for c in _trained if os.path.basename(os.path.dirname(c)) != "last"] or _trained


def _is_ckpt(p):
    return bool(p) and os.path.isdir(p) and os.path.exists(os.path.join(p, "config.json"))


if _explicit:
    POLICY_PATH = _explicit
    print("POLICY_PATH (from env):", POLICY_PATH)
elif _prefer_trained and _trained:
    POLICY_PATH = _trained[-1]
    print("PREFER_TRAINED=1 -> this run's Step-4 checkpoint:", POLICY_PATH)
elif _is_ckpt(REFERENCE):
    POLICY_PATH = REFERENCE
    print("POLICY_PATH -> our fine-tuned checkpoint (default):", POLICY_PATH)
elif _trained:
    POLICY_PATH = _trained[-1]
    print(f"reference not staged at {REFERENCE}; using this run's Step-4 checkpoint:", POLICY_PATH)
else:
    raise FileNotFoundError(
        "No fine-tuned checkpoint found.\n"
        f"  - expected our fine-tuned checkpoint at: {REFERENCE}\n"
        f"  - or a Step-4 output under: {OUTPUT_DIR}/checkpoints/*/pretrained_model\n"
        "Stage the fine-tuned checkpoint at REFERENCE_POLICY, or set POLICY_PATH=/path "
        "(or a Hub repo id), or re-run Step 4 with STEPS >= SAVE_FREQ."
    )

## 5. Load the LoRA checkpoint on the DROID base -> LIBERO policy (closed-loop eval)

The fine-tune saved a LeRobot-format checkpoint (`.../pretrained_model`): the DROID base config + the trained LoRA adapter weights + the LIBERO normalization/processor stats. Loading it with `--policy.path` reconstructs the runtime LIBERO policy (base + adapter merged at load), with no `norm_tag` needed. Here we run that policy in the LIBERO MuJoCo simulator (headless EGL on the AMD GPU) and report success.

**This is intentionally tiny.** A LIBERO suite has ~10 tasks and the evaluator runs `N_EPISODES` rollouts *per task*, so a whole suite is `N_EPISODES × 10` MuJoCo rollouts - minutes of stepping. For the workshop we pin a **single task** (`--env.task_ids=[TASK_ID]`) and run just a few episodes, so this cell finishes quickly and is only a smoke-number. a very short LoRA run won't converge, so use the reference checkpoint (last section) for a strong number and full-suite eval offline.

In [ ]:
# Closed-loop LIBERO eval of the checkpoint (headless EGL on the AMD GPU). The saved processor
# + normalization stats are restored from the checkpoint via --policy.path (no norm_tag). The
# exact lerobot-eval command is inline here; a very short LoRA run won't fully converge - use
# the reference checkpoint (last section) for a strong number.
SUITE = os.environ.get("SUITE", "libero_object")
# A LIBERO suite has ~10 tasks and the evaluator runs `n_episodes` rollouts PER task, so the
# full suite is n_episodes x 10. This is only a quick sanity number (the interactive sim is the
# real demo), so we pin ONE task (TASK_ID) and run a handful of episodes -> ~2-3 rollouts total.
TASK_ID = os.environ.get("TASK_ID", "3")
N_EPISODES = os.environ.get("N_EPISODES", "3")
SEED = os.environ.get("SEED", "1000")
RUN_DIR = os.path.join(OUT_DIR, f"_ft_eval_{SUITE}_t{TASK_ID}_seed{SEED}")
os.makedirs(RUN_DIR, exist_ok=True)

# lerobot disables its tqdm progress bars when inside_slurm() is true (it only checks for the
# SLURM_JOB_ID env var). Setting it here turns OFF the per-step rollout / eval-batch bars at the
# source; we then print one tidy line per episode from eval_info.json after inference finishes.
eval_env = child_env(MUJOCO_GL="egl", PYOPENGL_PLATFORM="egl", OMP_NUM_THREADS="1", MKL_NUM_THREADS="1", SLURM_JOB_ID="1")
cmd = [
    "/opt/train-venv/bin/lerobot-eval",
    f"--policy.path={POLICY_PATH}",
    "--policy.inference_action_mode=continuous",
    "--policy.model_dtype=bfloat16",
    "--policy.use_amp=true",
    "--policy.enable_inference_cuda_graph=false",
    "--policy.device=cuda",
    "--policy.per_episode_seed=true",
    f"--policy.eval_seed={SEED}",
    "--env.type=libero",
    f"--env.task={SUITE}",
    f"--env.task_ids=[{TASK_ID}]",  # single task -> keep the workshop eval to a few rollouts
    '--env.camera_name_mapping={"agentview_image":"image","robot0_eye_in_hand_image":"wrist_image"}',
    "--eval.batch_size=1",
    f"--eval.n_episodes={N_EPISODES}",
    f"--seed={SEED}",
    f"--output_dir={os.path.join(RUN_DIR, 'run')}",
]
print(f"quick closed-loop eval: suite={SUITE} task_id={TASK_ID}  ->  {N_EPISODES} rollout(s) total\n")

# Per-step bars are already off (SLURM_JOB_ID above); also hide lerobot's final raw metric-dict
# dump. A clean per-episode summary is printed from eval_info.json below.
def _eval_drop(line):
    return line.lstrip().startswith(("{", "[{", "Overall Aggregated Metrics", "Aggregated Metrics for"))

stream_cmd(cmd, env=eval_env, quiet=True, drop=_eval_drop)

# One line per episode (success + reward), then the aggregate.
_info = json.load(open(os.path.join(RUN_DIR, "run", "eval_info.json")))
print("\nclosed-loop results:")
_n = 0
for _t in _info.get("per_task", []):
    _m = _t.get("metrics", {})
    _succ, _sr = _m.get("successes", []), _m.get("sum_rewards", [])
    for _i, _ok in enumerate(_succ):
        _rew = f"{_sr[_i]:.2f}" if _i < len(_sr) else "n/a"
        print(f"  episode {_n + 1}: {'SUCCESS' if _ok else 'failure'}  (sum_reward={_rew})")
        _n += 1
_ov = _info.get("overall", {})
print(f"\nPASS: {_n} episode(s), success rate {_ov.get('pc_success', float('nan')):.1f}%  "
      f"(avg_sum_reward={_ov.get('avg_sum_reward', float('nan')):.2f}); artifacts in {RUN_DIR}")

## Reference checkpoint (compelling demo without waiting for a long train)

A short LoRA run demonstrates the *pipeline* but will not fully converge. For a strong sim demo, point `POLICY_PATH` at the provided longer-trained reference checkpoint (organizers supply the path / Hub repo; in this image it is placed under `~/checkpoints/reference/pretrained_model`), then re-run section 5:

```python
POLICY_PATH = os.path.expanduser("~/checkpoints/reference/pretrained_model")  # or a Hub repo id
```

Notes:
- Weights, images and large videos live only on the remote machine (workspace policy). Small eval artifacts land under `~/outputs`.
- To train for real, raise `STEPS` (e.g. 10000) and, on a multi-GPU Instinct node, `BATCH_SIZE`.

## Key Takeaways

Now you know:
- How **LoRA** fine-tunes a large vision-language-action model cheaply: freeze the SigLIP2 + Qwen3 backbone, train small low-rank adapters plus the flow-matching action expert
- The **real-to-sim** recipe: start from a policy trained on a real Franka arm and adapt it to the LIBERO simulator, which shares the embodiment and action space
- How to sanity-check a policy **open-loop** (predicted vs teleop actions) before spending time on training
- How to evaluate a fine-tuned policy **closed-loop** in the headless LIBERO MuJoCo simulator on AMD ROCm hardware
- Which knobs (`STEPS`, `BATCH_SIZE`, `FT_MODE`) turn a workshop smoke-test into a real training run

## What to Try Next

- Raise `STEPS` (e.g. `10000`) and watch both the loss curve and the closed-loop success rate improve
- Switch `FT_MODE` to `action_expert_only` or `full` and compare memory use and final quality
- Point the eval at a different LIBERO suite (`SUITE`) or task (`TASK_ID`) and see how the fine-tuned policy transfers
- Load the provided reference checkpoint and run the full LIBERO suite offline for a strong success number
- Continue to the interactive notebook and drive the fine-tuned policy in natural language

## References

* [MolmoAct2 base model (Hugging Face)](https://huggingface.co/allenai/MolmoAct2-DROID)
* [MolmoAct2 LIBERO dataset (Hugging Face)](https://huggingface.co/datasets/allenai/MolmoAct2-LIBERO-Dataset)
* [LeRobot](https://github.com/huggingface/lerobot)
* [LoRA: Low-Rank Adaptation of Large Language Models (arXiv:2106.09685)](https://arxiv.org/abs/2106.09685)

**Continue to**: [interactive_sim_molmoact2_libero.ipynb](interactive_sim_molmoact2_libero.ipynb)

---
Copyright© 2026 AMD, Inc SPDX-License-Identifier: MIT